Some questions in mind:
- Can the model become more accurate if we eliminate first time loaner? 
- If we divide our customer base like:
    - Entirely fresh customer: No credit bureau data, No home credit data
    - New to company, not new to credit: Have only the credit bureau data, haven't use any of the company product. 
    - Only engage with company: use only home credit product. 

# Load Libraries

In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

In [ ]:
# === CLASSIFIER IMPORTS ===
print("📦 Importing all required classifiers and libraries...")

# Core libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier

# Gradient boosting classifiers
try:
    from xgboost import XGBClassifier
    print("✅ XGBoost imported successfully")
except ImportError as e:
    print(f"❌ XGBoost import failed: {e}")
    XGBClassifier = None

try:
    from catboost import CatBoostClassifier
    print("✅ CatBoost imported successfully")
except ImportError as e:
    print(f"❌ CatBoost import failed: {e}")
    CatBoostClassifier = None



# Check which classifiers are available
available_classifiers = []
if XGBClassifier is not None:
    available_classifiers.append("XGBoost")
if CatBoostClassifier is not None:
    available_classifiers.append("CatBoost")


available_classifiers.append("Decision Tree")  # Always available

print(f"\n🎯 Available classifiers: {', '.join(available_classifiers)}")
print(f"✅ Import script completed!")


In [ ]:
# Configure Polars to show all rows and make it scrollable
pl.Config.set_tbl_rows(-1)  # Show all rows
pl.Config.set_tbl_cols(-1)  # Show all columns
pl.Config.set_tbl_width_chars(1000)  # Set wider table width

# Define Utils Function

In [ ]:
# Use sklearn models instead due to LightGBM installation issues
def train_and_evaluate_models_sklearn(baseline_data, full_data, test_size=0.2, random_state=42):
    """
    Train and evaluate both baseline and full models using sklearn models
    """
    from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
    from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
    from sklearn.model_selection import train_test_split
    
    results = {}
    
    # Prepare datasets
    datasets = {
        'baseline': baseline_data,
        'full': full_data
    }
    
    for dataset_name, data in datasets.items():
        print(f"\n=== TRAINING {dataset_name.upper()} MODEL ===")
        
        # Split features and target
        X = data.drop('TARGET', axis=1)
        y = data['TARGET']
        
        # Train-test split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state, stratify=y
        )
        
        print(f"Dataset: {data.shape}")
        print(f"Features: {X.shape[1]}")
        print(f"Train: {X_train.shape}, Test: {X_test.shape}")
        print(f"Target distribution: {y.value_counts(normalize=True).to_dict()}")
        
        # Train models
        models = {
            'Random Forest': RandomForestClassifier(
                n_estimators=100,
                random_state=random_state,
                n_jobs=-1
            ),
            'Gradient Boosting': GradientBoostingClassifier(
                n_estimators=100,
                random_state=random_state
            )
        }
        
        dataset_results = {}
        
        for model_name, model in models.items():
            print(f"\nTraining {model_name}...")
            
            # Train model
            model.fit(X_train, y_train)
            
            # Predictions
            y_pred_proba = model.predict_proba(X_test)[:, 1]
            y_pred = model.predict(X_test)
            
            # Evaluate
            auc_score = roc_auc_score(y_test, y_pred_proba)
            
            print(f"{model_name} AUC: {auc_score:.4f}")
            
            # Store results
            dataset_results[model_name] = {
                'model': model,
                'auc_score': auc_score,
                'y_test': y_test,
                'y_pred': y_pred,
                'y_pred_proba': y_pred_proba,
                'X_test': X_test
            }
            
            # Feature importance for tree-based models
            if hasattr(model, 'feature_importances_'):
                feature_importance = pd.DataFrame({
                    'feature': X.columns,
                    'importance': model.feature_importances_
                }).sort_values('importance', ascending=False)
                
                print(f"\nTop 10 important features:")
                print(feature_importance.head(10))
                
                dataset_results[model_name]['feature_importance'] = feature_importance
        
        results[dataset_name] = dataset_results
    
    return results

In [ ]:
# Updated version with LightGBM and Random Forest for better comparison
def train_and_evaluate_models_v2(baseline_data, full_data, test_size=0.2, random_state=42):
    """
    Train and evaluate both baseline and full models using LightGBM and Random Forest
    """
    import lightgbm as lgb
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
    from sklearn.model_selection import train_test_split
    
    results = {}
    
    # Prepare datasets
    datasets = {
        'baseline': baseline_data,
        'full': full_data
    }
    
    for dataset_name, data in datasets.items():
        print(f"\n=== TRAINING {dataset_name.upper()} MODEL ===")
        
        # Split features and target
        X = data.drop('TARGET', axis=1)
        y = data['TARGET']
        
        # Train-test split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state, stratify=y
        )
        
        print(f"Dataset: {data.shape}")
        print(f"Features: {X.shape[1]}")
        print(f"Train: {X_train.shape}, Test: {X_test.shape}")
        print(f"Target distribution: {y.value_counts(normalize=True).to_dict()}")
        
        # Train models
        models = {
            'LightGBM': lgb.LGBMClassifier(
                random_state=random_state,
                n_estimators=100,
                verbose=-1
            ),
            'Random Forest': RandomForestClassifier(
                n_estimators=100,
                random_state=random_state,
                n_jobs=-1
            )
        }
        
        dataset_results = {}
        
        for model_name, model in models.items():
            print(f"\nTraining {model_name}...")
            
            # Train model
            model.fit(X_train, y_train)
            
            # Predictions
            y_pred_proba = model.predict_proba(X_test)[:, 1]
            y_pred = model.predict(X_test)
            
            # Evaluate
            auc_score = roc_auc_score(y_test, y_pred_proba)
            
            print(f"{model_name} AUC: {auc_score:.4f}")
            
            # Store results
            dataset_results[model_name] = {
                'model': model,
                'auc_score': auc_score,
                'y_test': y_test,
                'y_pred': y_pred,
                'y_pred_proba': y_pred_proba,
                'X_test': X_test
            }
            
            # Feature importance for tree-based models
            if hasattr(model, 'feature_importances_'):
                feature_importance = pd.DataFrame({
                    'feature': X.columns,
                    'importance': model.feature_importances_
                }).sort_values('importance', ascending=False)
                
                print(f"\nTop 10 important features:")
                print(feature_importance.head(10))
                
                dataset_results[model_name]['feature_importance'] = feature_importance
        
        results[dataset_name] = dataset_results
    
    return results

In [ ]:
def prepare_full_dataset_for_modeling(df):
    """
    Prepare the full dataset for modeling - handle all features systematically
    """
    df_full = df.clone()
    
    # Remove ID column
    df_full = df_full.drop('SK_ID_CURR')
    
    # Handle missing values by feature type
    feature_stats = []
    
    for col in df_full.columns:
        if col == 'TARGET':
            continue
            
        col_type = df_full.schema[col]
        missing_count = df_full[col].null_count()
        missing_rate = missing_count / df_full.height
        
        if col_type in [pl.Float64, pl.Float32, pl.Int64, pl.Int32]:
            # Numerical: fill with median
            if missing_count > 0:
                median_val = df_full[col].median()
                df_full = df_full.with_columns(pl.col(col).fill_null(median_val))
                
        elif col_type == pl.Utf8:
            # Categorical: fill with 'MISSING' to preserve the signal
            if missing_count > 0:
                df_full = df_full.with_columns(pl.col(col).fill_null('MISSING'))
        
        feature_stats.append({
            'feature': col,
            'type': str(col_type),
            'missing_rate': missing_rate,
            'imputed': missing_count > 0
        })
    
    print(f"Full dataset prepared: {df_full.shape}")
    print(f"Features imputed: {sum(1 for stat in feature_stats if stat['imputed'])}")
    
    return df_full, feature_stats

def encode_categorical_features(df):
    """
    Encode categorical features for machine learning
    """
    from sklearn.preprocessing import LabelEncoder
    
    df_encoded = df.to_pandas()  # Convert to pandas for sklearn
    label_encoders = {}
    
    categorical_cols = [col for col in df_encoded.columns 
                       if df_encoded[col].dtype == 'object' and col != 'TARGET']
    
    print(f"Encoding {len(categorical_cols)} categorical features...")
    
    for col in categorical_cols:
        le = LabelEncoder()
        df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
        label_encoders[col] = le
    
    return df_encoded, label_encoders

In [ ]:
def calculate_feature_target_correlations(df):
    """
    Calculate correlations between all numerical features and the target variable.
    Handle missing values by using available data for correlation calculation.
    """
    
    # Get numerical columns (excluding ID and target)
    numerical_cols = [col for col, dtype in df.schema.items() 
                     if dtype in [pl.Float64, pl.Float32, pl.Int64, pl.Int32] 
                     and col not in ['SK_ID_CURR', 'TARGET']]
    
    correlations = []
    
    print(f"Calculating correlations for {len(numerical_cols)} numerical features...")
    
    # Convert to pandas for correlation calculation (more efficient for this task)
    df_pandas = df.select(['TARGET'] + numerical_cols).to_pandas()
    
    for col in numerical_cols:
        # Calculate correlation using available data (ignoring NaN)
        correlation = df_pandas['TARGET'].corr(df_pandas[col])
        
        if not pd.isna(correlation):  # Only include if correlation can be calculated
            missing_rate = df[col].null_count() / df.height
            correlations.append({
                'feature': col,
                'correlation': abs(correlation),  # Use absolute correlation for ranking
                'raw_correlation': correlation,
                'missing_rate': missing_rate
            })
    
    # Convert to DataFrame and sort by absolute correlation
    corr_df = pl.DataFrame(correlations).sort('correlation', descending=True)
    
    return corr_df

def analyze_categorical_target_relationship(df):
    """
    Analyze relationship between categorical features and target using contingency analysis
    """
    categorical_cols = [col for col, dtype in df.schema.items() 
                       if dtype == pl.Utf8 and col != 'SK_ID_CURR']
    
    cat_analysis = []
    
    for col in categorical_cols:
        if df[col].null_count() / df.height < 0.8:  # Only analyze if not too many missing
            # Calculate default rate by category
            analysis = df.group_by(col).agg([
                pl.count().alias('count'),
                pl.col('TARGET').mean().alias('default_rate')
            ]).filter(pl.col('count') > 100)  # Only categories with sufficient samples
            
            if analysis.height > 1:  # Need multiple categories to analyze
                default_rates = analysis['default_rate'].to_list()
                overall_default_rate = df['TARGET'].mean()
                
                # Calculate variation from overall rate
                max_deviation = max([abs(rate - overall_default_rate) for rate in default_rates])
                
                cat_analysis.append({
                    'feature': col,
                    'max_deviation': max_deviation,
                    'n_categories': analysis.height,
                    'missing_rate': df[col].null_count() / df.height
                })
    
    if cat_analysis:
        return pl.DataFrame(cat_analysis).sort('max_deviation', descending=True)
    else:
        return pl.DataFrame()

In [ ]:
def smart_feature_selection_and_handling(df):
    """
    Smart approach to handle features without domain knowledge:
    1. Start with low-missing, interpretable features
    2. Use feature importance to guide decisions about high-missing features
    3. Create simple baseline model first
    """
    
    # Separate features by missing rate and interpretability
    feature_analysis = []
    
    for col in df.columns:
        if col in ['SK_ID_CURR', 'TARGET']:
            continue
            
        missing_rate = df[col].null_count() / df.height
        is_normalized = any(suffix in col for suffix in ['_AVG', '_MODE', '_MEDI', '_NORMALIZED'])
        is_external = col.startswith('EXT_SOURCE')
        is_flag = col.startswith('FLAG_')
        is_interpretable = col in ['AMT_CREDIT', 'AMT_INCOME_TOTAL', 'AMT_ANNUITY', 
                                 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'CODE_GENDER', 
                                 'CNT_CHILDREN', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS']
        
        feature_analysis.append({
            'column': col,
            'missing_rate': missing_rate,
            'is_normalized': is_normalized,
            'is_external': is_external,
            'is_flag': is_flag,
            'is_interpretable': is_interpretable
        })
    
    feature_df = pl.DataFrame(feature_analysis).sort('missing_rate')
    
    # Strategy 1: Start with highly interpretable, low-missing features
    core_features = feature_df.filter(
        (pl.col('is_interpretable') == True) | 
        ((pl.col('missing_rate') < 0.1) & (pl.col('is_flag') == True))
    )['column'].to_list()
    
    # Strategy 2: Add external scores (known to be important in credit scoring)
    external_features = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
    
    # Strategy 3: Add some low-missing, non-normalized features
    stable_features = feature_df.filter(
        (pl.col('missing_rate') < 0.05) & 
        (pl.col('is_normalized') == False) &
        (~pl.col('column').str.contains('DOCUMENT'))  # Skip document flags for now
    )['column'].to_list()
    
    selected_features = list(set(core_features + external_features + stable_features))
    
    print(f"Core interpretable features: {len(core_features)}")
    print(f"External score features: {len(external_features)}")  
    print(f"Stable low-missing features: {len(stable_features)}")
    print(f"Total selected for baseline model: {len(selected_features)}")
    
    return selected_features, feature_df

def create_baseline_dataset(df, selected_features):
    """Create a clean baseline dataset for initial modeling"""
    
    # Select features + target
    baseline_features = selected_features + ['TARGET']
    df_baseline = df.select(baseline_features)
    
    # Simple imputation strategy
    df_clean = df_baseline.clone()
    
    # Handle numerical missing values with median
    for col in selected_features:
        col_type = df_baseline.schema[col]
        if col_type in [pl.Float64, pl.Float32, pl.Int64, pl.Int32]:
            if df_clean[col].null_count() > 0:
                median_val = df_clean[col].median()
                df_clean = df_clean.with_columns(pl.col(col).fill_null(median_val))
                
        # Handle categorical missing values with mode
        elif col_type == pl.Utf8:
            if df_clean[col].null_count() > 0:
                mode_val = df_clean[col].mode().first()
                if mode_val is None:
                    mode_val = 'Unknown'
                df_clean = df_clean.with_columns(pl.col(col).fill_null(mode_val))
    
    print(f"\nBaseline dataset shape: {df_clean.shape}")
    print(f"Features with missing values after imputation: {df_clean.null_count().sum_horizontal()[0]}")
    
    return df_clean

In [ ]:
def get_missing_data_summary(df):
    """
    Create a summary dataframe showing missing data statistics for each column.
    
    Args:
        df: Polars DataFrame
        
    Returns:
        Polars DataFrame with columns: column_name, missing_count, missing_proportion
    """
    total_rows = df.height
    
    missing_summary = pl.DataFrame({
        'column_name': df.columns,
        'missing_count': [df[col].null_count() for col in df.columns],
        'missing_proportion': [round(df[col].null_count() / total_rows, 4) for col in df.columns]
    }).sort('missing_proportion', descending=True)
    
    return missing_summary

In [ ]:
def handle_grouped_missing_features(df):
    """
    Handle features that are missing together in groups (e.g., AVG, MODE, MEDI variants).
    
    Strategy:
    1. Identify feature groups (same prefix before _AVG, _MODE, _MEDI)
    2. For each group, create a single representative feature or impute all together
    3. Create missing indicators for groups with high missing rates
    
    Args:
        df: Polars DataFrame
        
    Returns:
        Processed DataFrame
    """
    df_processed = df.clone()
    
    # Find feature groups by prefix
    feature_groups = {}
    for col in df.columns:
        if any(suffix in col for suffix in ['_AVG', '_MODE', '_MEDI']):
            # Extract the base name (everything before _AVG/_MODE/_MEDI)
            base_name = col.replace('_AVG', '').replace('_MODE', '').replace('_MEDI', '')
            if base_name not in feature_groups:
                feature_groups[base_name] = []
            feature_groups[base_name].append(col)
    
    print(f"Found {len(feature_groups)} feature groups:")
    for base_name, cols in feature_groups.items():
        if len(cols) > 1:  # Only show groups with multiple features
            missing_rate = df[cols[0]].null_count() / df.height
            print(f"  {base_name}: {len(cols)} features, {missing_rate:.3f} missing rate")
    
    return df_processed, feature_groups

def create_grouped_features(df, feature_groups, missing_threshold=0.5):
    """
    Create consolidated features from grouped missing features.
    
    Strategies:
    1. If missing rate < threshold: Use mean of available values, impute with group median
    2. If missing rate >= threshold: Create binary indicator + drop original features
    3. For some groups: Create single representative feature (e.g., use _AVG if available)
    """
    df_processed = df.clone()
    
    for base_name, cols in feature_groups.items():
        if len(cols) <= 1:
            continue
            
        missing_rate = df[cols[0]].null_count() / df.height
        
        if missing_rate >= missing_threshold:
            # High missing rate: create binary indicator
            indicator_col = f"{base_name}_AVAILABLE"
            df_processed = df_processed.with_columns(
                pl.when(pl.col(cols[0]).is_not_null()).then(1).otherwise(0).alias(indicator_col)
            )
            print(f"Created indicator {indicator_col} for {base_name} (missing rate: {missing_rate:.3f})")
            
        else:
            # Low missing rate: create consolidated feature
            # Use mean of AVG, MODE, MEDI where available, then impute
            avg_col = [c for c in cols if '_AVG' in c]
            if avg_col:
                # Use AVG as representative, impute with median
                consolidated_col = f"{base_name}_CONSOLIDATED"
                median_val = df[avg_col[0]].median()
                df_processed = df_processed.with_columns(
                    pl.col(avg_col[0]).fill_null(median_val).alias(consolidated_col)
                )
                print(f"Created consolidated feature {consolidated_col} (imputed with median: {median_val})")
    
    return df_processed

# Load Data

In [ ]:
# Load all data files using Polars
print("Loading datasets with Polars...")

# Main application data
train_df = pl.read_csv('../data/application_train.csv')
test_df = pl.read_csv('../data/application_test.csv')

# Additional data sources
bureau_df = pl.read_csv('../data/bureau.csv')
bureau_balance_df = pl.read_csv('../data/bureau_balance.csv')
previous_app_df = pl.read_csv('../data/previous_application.csv')
credit_card_df = pl.read_csv('../data/credit_card_balance.csv')
pos_cash_df = pl.read_csv('../data/POS_CASH_balance.csv')
pos_cash_df_preprocessed = pl.read_csv('../data/POS_CASH_balance_processing.csv')
installments_df = pl.read_csv('../data/installments_payments.csv')
bureau_risk_df = pl.read_csv('../data/bureau_risk_features.csv')

print(f"Training data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")
print(f"Bureau data shape: {bureau_df.shape}")
print(f"Previous applications shape: {previous_app_df.shape}")
print(f"Credit card balance shape: {credit_card_df.shape}")
print(f"POS Cash balance shape: {pos_cash_df.shape}")
print(f"Installments payments shape: {installments_df.shape}")

print(f"\nTarget distribution:")
target_counts = train_df.select(pl.col('TARGET')).to_pandas()['TARGET'].value_counts(normalize=True)
print(target_counts)

# Simple EDA

In [ ]:
# Explore the main training data structure
print("Training data column info:")
print(f"Number of columns: {len(train_df.columns)}")
print(f"Column names: {train_df.columns[:20]}...")  # First 20 columns

print("\nData types:")
print(train_df.schema)

print("\nBasic statistics for numerical columns:")
numerical_cols = [col for col, dtype in train_df.schema.items() if dtype in [pl.Float64, pl.Float32, pl.Int64, pl.Int32]]
print(f"Found {len(numerical_cols)} numerical columns")

# Display first few rows
print("\nFirst 5 rows:")
print(train_df.head())

In [ ]:
# Apply the function to our training data
missing_summary = get_missing_data_summary(train_df)
print("Missing data summary (top 20 columns by missing proportion):")
print(missing_summary)

In [ ]:
# Apply the functions
print("Analyzing grouped missing features...")
df_processed, feature_groups = handle_grouped_missing_features(train_df)

print("\nCreating consolidated features...")
df_processed = create_grouped_features(train_df, feature_groups, missing_threshold=0.5)

In [ ]:
# Apply smart feature selection
print("Analyzing features for baseline model...")
selected_features, feature_analysis = smart_feature_selection_and_handling(train_df)

print(f"\nSelected features for baseline model:")
for i, feat in enumerate(selected_features[:15]):  # Show first 15
    print(f"  {i+1:2d}. {feat}")
if len(selected_features) > 15:
    print(f"  ... and {len(selected_features) - 15} more")

# Create baseline dataset
baseline_train = create_baseline_dataset(train_df, selected_features)

In [ ]:
# Calculate correlations
print("=== FEATURE-TARGET CORRELATION ANALYSIS ===")
correlation_results = calculate_feature_target_correlations(train_df)

print(f"\nTop 20 features by correlation with TARGET:")
print(correlation_results.head(20))

print(f"\nTop features with high correlation AND low missing rate:")
high_quality_features = correlation_results.filter(
    (pl.col('correlation') > 0.05) & (pl.col('missing_rate') < 0.1)
)
print(high_quality_features.head(10))

# Analyze categorical features  
print(f"\n=== CATEGORICAL FEATURE ANALYSIS ===")
categorical_analysis = analyze_categorical_target_relationship(train_df)
if categorical_analysis.height > 0:
    print(f"\nTop categorical features by variation in default rates:")
    print(categorical_analysis.head(10))

In [ ]:
import pandas as pd  # Add pandas import for correlation calculation

# Re-run the correlation analysis
print("=== FEATURE-TARGET CORRELATION ANALYSIS ===")
correlation_results = calculate_feature_target_correlations(train_df)

print(f"\nTop 20 features by correlation with TARGET:")
print(correlation_results.head(20))

print(f"\nFeatures with correlation > 0.1 (strong predictors):")
strong_features = correlation_results.filter(pl.col('correlation') > 0.1)
print(strong_features)

print(f"\nHigh-missing features that still show strong correlation:")
strong_but_missing = correlation_results.filter(
    (pl.col('correlation') > 0.05) & (pl.col('missing_rate') > 0.3)
)
print("These might be worth keeping despite high missing rates:")
print(strong_but_missing)

# Basic Modeling

In [ ]:

# Prepare both datasets
print("=== PREPARING DATASETS FOR MODELING ===")

# 1. Baseline dataset (selected features only)
print("\n1. Preparing baseline dataset...")
if 'baseline_train' not in locals():
    baseline_train = create_baseline_dataset(train_df, selected_features)

baseline_pandas = baseline_train.to_pandas()
print(f"Baseline dataset: {baseline_pandas.shape}")

# 2. Full dataset (all features)
print("\n2. Preparing full dataset...")
full_train, feature_stats = prepare_full_dataset_for_modeling(train_df)
full_encoded, encoders = encode_categorical_features(full_train)
print(f"Full dataset: {full_encoded.shape}")

# Show some statistics
print(f"\nDataset comparison:")
print(f"Baseline features: {baseline_pandas.shape[1] - 1}")  # -1 for TARGET
print(f"Full features: {full_encoded.shape[1] - 1}")
print(f"Feature reduction: {1 - (baseline_pandas.shape[1] - 1) / (full_encoded.shape[1] - 1):.2%}")

In [ ]:

# Prepare both datasets
print("=== PREPARING DATASETS FOR MODELING ===")

# 1. Baseline dataset (selected features only)
print("\n1. Preparing baseline dataset...")
if 'baseline_train' not in locals():
    baseline_train = create_baseline_dataset(train_df, selected_features)

baseline_pandas = baseline_train.to_pandas()
print(f"Baseline dataset: {baseline_pandas.shape}")

# 2. Full dataset (all features)
print("\n2. Preparing full dataset...")
full_train, feature_stats = prepare_full_dataset_for_modeling(train_df)
full_encoded, encoders = encode_categorical_features(full_train)
print(f"Full dataset: {full_encoded.shape}")

# Show some statistics
print(f"\nDataset comparison:")
print(f"Baseline features: {baseline_pandas.shape[1] - 1}")  # -1 for TARGET
print(f"Full features: {full_encoded.shape[1] - 1}")
print(f"Feature reduction: {1 - (baseline_pandas.shape[1] - 1) / (full_encoded.shape[1] - 1):.2%}")

In [ ]:
# Fix categorical encoding issue before training
print("=== FIXING CATEGORICAL ENCODING ISSUES ===")
print("Checking for categorical features in baseline dataset...")

# Check data types in baseline_pandas
print("\nBaseline dataset dtypes:")
print(baseline_pandas.dtypes.value_counts())

# Identify categorical columns
categorical_cols = baseline_pandas.select_dtypes(include=['object']).columns.tolist()
if 'TARGET' in categorical_cols:
    categorical_cols.remove('TARGET')

print(f"\nCategorical columns found: {categorical_cols}")

# Encode categorical features in baseline dataset
if len(categorical_cols) > 0:
    from sklearn.preprocessing import LabelEncoder
    baseline_pandas_encoded = baseline_pandas.copy()
    
    for col in categorical_cols:
        print(f"Encoding {col}...")
        le = LabelEncoder()
        baseline_pandas_encoded[col] = le.fit_transform(baseline_pandas_encoded[col].astype(str))
    
    print("Categorical encoding completed for baseline dataset.")
else:
    baseline_pandas_encoded = baseline_pandas.copy()
    print("No categorical columns found in baseline dataset.")

# Check final dtypes
print(f"\nFinal baseline dataset shape: {baseline_pandas_encoded.shape}")
print("Final dtypes:")
print(baseline_pandas_encoded.dtypes.value_counts())

# Train models with sklearn
print("\n=== TRAINING MODELS WITH SKLEARN ===")
# model_results_sklearn = train_and_evaluate_models_sklearn(baseline_pandas_encoded, full_encoded)

## Feature Engineering Approaches Used

### 1. Missing Data Analysis Approach
- **Identified grouped features** (AVG/MODE/MEDI variants) that were missing together
- **Example**: `COMMONAREA_AVG`, `COMMONAREA_MODE`, `COMMONAREA_MEDI` all had ~70% missing rate
- These represent **external data** that may not be available for all properties.
- Some features tends to be missing together, this maybe because these are derived from a non-neccessary feature

### 2. Smart Feature Selection Strategy

#### Baseline Model: Selected interpretable, low-missing features
- **Core demographic**: `AMT_INCOME_TOTAL`, `DAYS_BIRTH`, `CODE_GENDER`, `CNT_CHILDREN`
- **Financial metrics**: `AMT_CREDIT`, `AMT_ANNUITY`, `AMT_GOODS_PRICE`
- **External credit scores**: `EXT_SOURCE_1`, `EXT_SOURCE_2`, `EXT_SOURCE_3`
- **Contact info flags**: `FLAG_*` variables

#### Full Model: Used ALL features with systematic imputation
- **Numerical features**: Imputed with median
- **Categorical features**: Created `'MISSING'` category to preserve signal

### 3. Correlation-Based Feature Ranking
- Calculated correlations between each feature and `TARGET` variable
- Identified high-value features even among high-missing ones
- Let the data guide feature importance decisions

### 4. Categorical Encoding
- **Label encoding** for tree-based models (they can handle this well)
- **Preserved `'MISSING'`** as a distinct category rather than imputing

## Key Insights from Results

### Why External Credit Scores (EXT_SOURCE_*) Ranked as Top Features:
- 🏛️ **Professional credit assessments** from external agencies
- 📊 **Capture creditworthiness information** not available in application data
- 🎯 **Specifically designed** for credit risk prediction
- 💪 **Strong predictive signal** even with some missing values

### However, there are some gotchas
- Not all customer have these external credit scores, therefore these can be null as well.
- Some of the missing data pattern can be informative (i.e the information about the customer's living area can be missing from our database or they not provide those information at all).

## Feature Engineering Lessons

### ✅ What Worked:
- Simple, interpretable features often outperform complex engineered ones
- External/third-party data sources are extremely valuable in credit scoring
- Missing data patterns can be informative (building info missing = certain property types)
- Domain-specific features (credit scores) beat generic demographic features

### ❌ What Didn't Work:
- High-missing normalized building features added more noise than signal
- Too many features can hurt model performance (curse of dimensionality)

## Business Implications
- 🤝 **Focus data acquisition efforts** on external credit bureau relationships
- 🎯 **Prioritize features** that are consistently available and high-quality
- 🔄 **Don't over-engineer features** when simple ones work well

# Try with preprocessed pos_processes

In [ ]:
# Check what columns are available in our datasets
print("=== CHECKING DATASET COLUMNS ===")
print(f"Train DF columns: {list(train_df.columns)[:10]}...")
print(f"Baseline train columns: {list(baseline_train.columns)[:10]}...")
print(f"POS features columns: {list(pos_cash_df_preprocessed.columns)[:10]}...")

# Check if SK_ID_CURR exists
print(f"\nSK_ID_CURR in train_df: {'SK_ID_CURR' in train_df.columns}")
print(f"SK_ID_CURR in baseline_train: {'SK_ID_CURR' in baseline_train.columns}")
print(f"SK_ID_CURR in pos_cash_df_preprocessed: {'SK_ID_CURR' in pos_cash_df_preprocessed.columns}")

# The issue is that baseline_train was created without SK_ID_CURR
# Let's recreate the baseline dataset with SK_ID_CURR included
print(f"\n=== RECREATING BASELINE DATASET WITH SK_ID_CURR ===")

def create_baseline_dataset_with_id(df, selected_features):
    """Create a clean baseline dataset for initial modeling, keeping SK_ID_CURR"""
    
    # Select features + target + ID
    baseline_features = selected_features + ['TARGET', 'SK_ID_CURR']
    df_baseline = df.select(baseline_features)
    
    # Simple imputation strategy
    df_clean = df_baseline.clone()
    
    # Handle numerical missing values with median
    for col in selected_features:
        col_type = df_baseline.schema[col]
        if col_type in [pl.Float64, pl.Float32, pl.Int64, pl.Int32]:
            if df_clean[col].null_count() > 0:
                median_val = df_clean[col].median()
                df_clean = df_clean.with_columns(pl.col(col).fill_null(median_val))
                
        # Handle categorical missing values with mode
        elif col_type == pl.Utf8:
            if df_clean[col].null_count() > 0:
                mode_val = df_clean[col].mode().first()
                if mode_val is None:
                    mode_val = 'Unknown'
                df_clean = df_clean.with_columns(pl.col(col).fill_null(mode_val))
    
    print(f"Baseline dataset shape: {df_clean.shape}")
    print(f"Features with missing values after imputation: {df_clean.null_count().sum_horizontal()[0]}")
    
    return df_clean

# Recreate baseline dataset with SK_ID_CURR
baseline_train_with_id = create_baseline_dataset_with_id(train_df, selected_features)

# Also need to ensure full_train has SK_ID_CURR 
print(f"Full train has SK_ID_CURR: {'SK_ID_CURR' in full_train.columns}")
if 'SK_ID_CURR' not in full_train.columns:
    print("Adding SK_ID_CURR to full_train...")
    full_train = train_df.select(['SK_ID_CURR'] + [col for col in full_train.columns if col != 'SK_ID_CURR'])
    print(f"Full train updated shape: {full_train.shape}")

In [ ]:
# Now join POS features with the corrected datasets
def join_pos_features_with_dataset(base_df, pos_features_df, fill_strategy='smart'):
    """
    Join POS features with base dataset, handling missing customers intelligently.
    
    Args:
        base_df: Base dataset (Polars DataFrame)
        pos_features_df: POS features dataset (Polars DataFrame)
        fill_strategy: How to handle missing POS customers ('zero', 'smart')
    """
    
    # Join datasets
    joined_df = base_df.join(pos_features_df, on='SK_ID_CURR', how='left')
    
    # Get POS feature columns (exclude SK_ID_CURR)
    pos_cols = [col for col in pos_features_df.columns if col != 'SK_ID_CURR']
    
    print(f"Joined dataset shape: {joined_df.shape}")
    
    # Count missing values in POS features after join
    missing_customers = joined_df[pos_cols[0]].null_count()
    print(f"Customers without POS history: {missing_customers:,} ({missing_customers/joined_df.shape[0]*100:.1f}%)")
    
    # Handle missing values based on strategy
    if fill_strategy == 'smart':
        # Smart filling: 0 for counts/flags, appropriate values for others
        count_flag_cols = [col for col in pos_cols if any(x in col for x in ['COUNT', 'FLAG', 'TOTAL'])]
        rate_ratio_cols = [col for col in pos_cols if any(x in col for x in ['RATE', 'RATIO', 'MEAN'])]
        other_cols = [col for col in pos_cols if col not in count_flag_cols + rate_ratio_cols]
        
        print(f"Smart fill strategy:")
        print(f"  - Count/Flag columns ({len(count_flag_cols)}): Fill with 0")
        print(f"  - Rate/Ratio columns ({len(rate_ratio_cols)}): Fill with median")
        print(f"  - Other columns ({len(other_cols)}): Fill with 0 or appropriate value")
        
        # Fill counts/flags with 0 (no history)
        for col in count_flag_cols:
            joined_df = joined_df.with_columns(pl.col(col).fill_null(0))
        
        # Fill rates with median of existing customers (or 0 if all null)
        for col in rate_ratio_cols:
            median_val = joined_df[col].median()
            fill_val = median_val if median_val is not None else 0.0
            joined_df = joined_df.with_columns(pl.col(col).fill_null(fill_val))
        
        # Fill other columns appropriately
        for col in other_cols:
            if 'RISK_SEGMENT' in col:
                joined_df = joined_df.with_columns(pl.col(col).fill_null('NEW_CUSTOMER'))
            elif 'MONTH' in col:
                joined_df = joined_df.with_columns(pl.col(col).fill_null(0))
            else:
                joined_df = joined_df.with_columns(pl.col(col).fill_null(0))
    
    return joined_df

print("\n=== JOINING POS FEATURES WITH CORRECTED DATASETS ===")

# 1. Join with baseline dataset (now with SK_ID_CURR)
baseline_with_pos = join_pos_features_with_dataset(
    baseline_train_with_id, pos_cash_df_preprocessed, fill_strategy='smart'
)

print(f"\nBaseline + POS shape: {baseline_with_pos.shape}")
print(f"New POS features added: {baseline_with_pos.shape[1] - baseline_train_with_id.shape[1]}")

# 2. Join with full dataset (add SK_ID_CURR if missing)
if 'SK_ID_CURR' not in full_train.columns:
    # Add SK_ID_CURR to full_train
    full_train_with_id = full_train.with_columns(train_df.select('SK_ID_CURR'))
else:
    full_train_with_id = full_train

full_with_pos = join_pos_features_with_dataset(
    full_train_with_id, pos_cash_df_preprocessed, fill_strategy='smart'
)

print(f"\nFull + POS shape: {full_with_pos.shape}")
print(f"New POS features added: {full_with_pos.shape[1] - full_train_with_id.shape[1]}")

In [ ]:
baseline_with_pos["SK_ID_CURR"].head()

In [ ]:
# Prepare enhanced datasets for modeling
print("\n=== PREPARING ENHANCED DATASETS FOR MODELING ===")

# Remove SK_ID_CURR before modeling (we don't want to train on ID)
baseline_pos_for_ml = baseline_with_pos.drop('SK_ID_CURR').to_pandas()
full_pos_for_ml = full_with_pos.drop('SK_ID_CURR')

# Check for missing values after join
print(f"Baseline + POS missing values: {baseline_pos_for_ml.isnull().sum().sum()}")
print(f"Full + POS missing values: {full_pos_for_ml.null_count().sum_horizontal()[0]}")

# Fix missing values in baseline + POS dataset
if baseline_pos_for_ml.isnull().sum().sum() > 0:
    print("Fixing missing values in baseline + POS dataset...")
    
    # Fill numerical columns with median
    numeric_cols = baseline_pos_for_ml.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if baseline_pos_for_ml[col].isnull().sum() > 0:
            median_val = baseline_pos_for_ml[col].median()
            baseline_pos_for_ml[col].fillna(median_val, inplace=True)
    
    # Fill categorical columns with mode or 'Unknown'
    categorical_cols = baseline_pos_for_ml.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        if baseline_pos_for_ml[col].isnull().sum() > 0:
            mode_val = baseline_pos_for_ml[col].mode()
            fill_val = mode_val[0] if len(mode_val) > 0 else 'Unknown'
            baseline_pos_for_ml[col].fillna(fill_val, inplace=True)
    
    print(f"After imputation - missing values: {baseline_pos_for_ml.isnull().sum().sum()}")

# Handle categorical encoding for baseline + POS
categorical_cols_baseline = baseline_pos_for_ml.select_dtypes(include=['object']).columns
if 'TARGET' in categorical_cols_baseline:
    categorical_cols_baseline = categorical_cols_baseline.drop('TARGET')

if len(categorical_cols_baseline) > 0:
    from sklearn.preprocessing import LabelEncoder
    print(f"Encoding {len(categorical_cols_baseline)} categorical features in baseline + POS...")
    
    for col in categorical_cols_baseline:
        le = LabelEncoder()
        baseline_pos_for_ml[col] = le.fit_transform(baseline_pos_for_ml[col].astype(str))

# Handle categorical encoding for full + POS
full_pos_encoded, pos_encoders = encode_categorical_features(full_pos_for_ml)

# Final verification - check for any remaining NaN values
print(f"\nFinal verification:")
print(f"Baseline + POS NaN count: {baseline_pos_for_ml.isnull().sum().sum()}")
print(f"Baseline + POS data types: {baseline_pos_for_ml.dtypes.value_counts()}")
print(f"Full + POS NaN count: {full_pos_encoded.isnull().sum().sum()}")

print(f"\nDataset comparison:")
print(f"Original Baseline: {baseline_pandas.shape[1] - 1} features")
print(f"Baseline + POS: {baseline_pos_for_ml.shape[1] - 1} features")
print(f"Original Full: {full_encoded.shape[1] - 1} features") 
print(f"Full + POS: {full_pos_encoded.shape[1] - 1} features")

pos_feature_count = pos_cash_df_preprocessed.shape[1] - 1  # -1 for SK_ID_CURR
print(f"\nPOS features impact: Added {pos_feature_count} new features to both datasets")

# Try with preprocessed credit bureau feature

In [ ]:
# ============================================================================
# ADDING BUREAU RISK FEATURES TO BASELINE + POS MODEL
# ============================================================================
print("=== INTEGRATING BUREAU RISK FEATURES ===")

# Step 1: Load the bureau risk features from our EDA notebook
try:
    # Try to load the saved bureau features first
    bureau_risk_features = pl.read_csv('../data/bureau_risk_features.csv')
    print(f"✅ Loaded saved bureau risk features: {bureau_risk_features.shape}")
except:
    print("⚠️ Saved bureau features not found. Creating from scratch...")
    
    # Create bureau risk features from scratch (same as in EDA notebook)
    # Basic bureau risk features
    bureau_risk_features = bureau_df.group_by('SK_ID_CURR').agg([
        # Basic counts and diversity
        pl.count().alias('BUREAU_TOTAL_COUNT'),
        pl.col('CREDIT_TYPE').n_unique().alias('BUREAU_CREDIT_TYPES_COUNT'),
        
        # Credit status risk indicators
        (pl.col('CREDIT_ACTIVE') == 'Active').sum().alias('BUREAU_ACTIVE_COUNT'),
        (pl.col('CREDIT_ACTIVE') == 'Closed').sum().alias('BUREAU_CLOSED_COUNT'),
        (pl.col('CREDIT_ACTIVE') == 'Bad debt').sum().alias('BUREAU_BAD_DEBT_COUNT'),
        (pl.col('CREDIT_ACTIVE') == 'Sold').sum().alias('BUREAU_SOLD_COUNT'),
        
        # High-risk status ratios
        ((pl.col('CREDIT_ACTIVE') == 'Bad debt').sum() / pl.count()).alias('BUREAU_BAD_DEBT_RATIO'),
        ((pl.col('CREDIT_ACTIVE') == 'Sold').sum() / pl.count()).alias('BUREAU_SOLD_RATIO'),
        (((pl.col('CREDIT_ACTIVE') == 'Bad debt') | (pl.col('CREDIT_ACTIVE') == 'Sold')).sum() / pl.count()).alias('BUREAU_HIGH_RISK_RATIO'),
        
        # Overdue risk indicators
        pl.col('CREDIT_DAY_OVERDUE').sum().alias('BUREAU_OVERDUE_DAYS_TOTAL'),
        pl.col('CREDIT_DAY_OVERDUE').mean().alias('BUREAU_OVERDUE_DAYS_MEAN'),
        pl.col('CREDIT_DAY_OVERDUE').max().alias('BUREAU_OVERDUE_DAYS_MAX'),
        (pl.col('CREDIT_DAY_OVERDUE') > 0).sum().alias('BUREAU_OVERDUE_COUNT'),
        ((pl.col('CREDIT_DAY_OVERDUE') > 0).sum() / pl.count()).alias('BUREAU_OVERDUE_RATIO'),
        
        # Amount-based overdue risk
        pl.col('AMT_CREDIT_SUM_OVERDUE').sum().alias('BUREAU_AMT_OVERDUE_TOTAL'),
        pl.col('AMT_CREDIT_SUM_OVERDUE').mean().alias('BUREAU_AMT_OVERDUE_MEAN'),
        (pl.col('AMT_CREDIT_SUM_OVERDUE') > 0).sum().alias('BUREAU_AMT_OVERDUE_COUNT'),
        ((pl.col('AMT_CREDIT_SUM_OVERDUE') > 0).sum() / pl.count()).alias('BUREAU_AMT_OVERDUE_RATIO'),
        
        # Credit utilization and debt ratios  
        (pl.col('AMT_CREDIT_SUM_DEBT').sum() / pl.col('AMT_CREDIT_SUM').sum()).alias('BUREAU_DEBT_TO_CREDIT_RATIO'),
        (pl.col('AMT_CREDIT_SUM_OVERDUE').sum() / pl.col('AMT_CREDIT_SUM').sum()).alias('BUREAU_OVERDUE_TO_CREDIT_RATIO'),
    ])
    
    # Save for future use
    bureau_risk_features.write_csv('../data/bureau_risk_features.csv')
    print(f"✅ Created and saved bureau risk features: {bureau_risk_features.shape}")

print(f"Bureau risk features: {len(bureau_risk_features.columns) - 1} features for {bureau_risk_features.shape[0]:,} clients")

In [ ]:
# Step 2: Create enhanced datasets with Bureau features
print("\n=== CREATING BASELINE + POS + BUREAU DATASET ===")

# Convert bureau features to pandas for consistency with existing pipeline
bureau_risk_pandas = bureau_risk_features.to_pandas()

# Join bureau features with existing baseline+POS dataset
# First, add SK_ID_CURR back to baseline_pos_clean for joining
baseline_pos_with_id = baseline_with_pos.select(['SK_ID_CURR'] + [col for col in baseline_with_pos.columns if col != 'SK_ID_CURR'])
baseline_pos_with_bureau = baseline_pos_with_id.join(bureau_risk_features, on='SK_ID_CURR', how='left')

print(f"Original Baseline + POS shape: {baseline_with_pos.shape}")
print(f"With Bureau features added: {baseline_pos_with_bureau.shape}")
print(f"New bureau features added: {baseline_pos_with_bureau.shape[1] - baseline_with_pos.shape[1]}")

# Handle missing values for clients without bureau data
bureau_feature_cols = [col for col in bureau_risk_features.columns if col != 'SK_ID_CURR']
print(f"\nFilling missing values for {len(bureau_feature_cols)} bureau features...")

# Convert to pandas and fill missing bureau features with 0 (no bureau history)
baseline_pos_bureau_pandas = baseline_pos_with_bureau.to_pandas()

for col in bureau_feature_cols:
    if col in baseline_pos_bureau_pandas.columns:
        missing_count = baseline_pos_bureau_pandas[col].isnull().sum()
        if missing_count > 0:
            baseline_pos_bureau_pandas[col].fillna(0, inplace=True)
            print(f"  {col}: filled {missing_count:,} missing values with 0")

# Remove SK_ID_CURR for modeling
baseline_pos_bureau_for_ml = baseline_pos_bureau_pandas.drop('SK_ID_CURR', axis=1)

print(f"\n✅ Final dataset ready for modeling:")
print(f"   Shape: {baseline_pos_bureau_for_ml.shape}")
print(f"   Features: {baseline_pos_bureau_for_ml.shape[1] - 1} (excluding TARGET)")
print(f"   Missing values: {baseline_pos_bureau_for_ml.isnull().sum().sum()}")

# Breakdown of feature types
original_features = train_df.shape[1] - 1  # -1 for TARGET
pos_features = baseline_with_pos.shape[1] - train_df.shape[1]  # New POS features
bureau_features = len(bureau_feature_cols)

print(f"\nFeature breakdown:")
print(f"   Original application features: {original_features}")
print(f"   POS cash balance features: {pos_features}")
print(f"   Bureau risk features: {bureau_features}")
print(f"   Total features: {original_features + pos_features + bureau_features}")

In [ ]:
# === TRAINING ENHANCED MODELS (XGBoost + CatBoost + GradientBoosting) ===
print("=== TRAINING ENHANCED MODELS (XGBoost + CatBoost + GradientBoosting) ===")

import time
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder

# Try to import CatBoost
try:
    from catboost import CatBoostClassifier
    catboost_available = True
    print("✅ CatBoost imported successfully")
except ImportError:
    print("⚠️  CatBoost not available. Install with: pip install catboost")
    catboost_available = False

try:
    # =============================
    # Preprocessing
    # =============================
    print("🔧 Preprocessing data and handling infinite values...")

    model_data = baseline_pos_bureau_for_ml.copy()

    # Handle categorical columns
    categorical_cols = model_data.select_dtypes(include=['object']).columns.tolist()
    if 'TARGET' in categorical_cols:
        categorical_cols.remove('TARGET')

    print(f"Encoding {len(categorical_cols)} categorical columns...")

    original_categorical_cols = categorical_cols.copy()  # Keep original for CatBoost
    label_encoders = {}
    for col in categorical_cols:
        le = LabelEncoder()
        model_data[col] = model_data[col].fillna('Unknown').astype(str)
        model_data[col] = le.fit_transform(model_data[col])
        label_encoders[col] = le

    # Handle infinite & extreme values
    print("🔍 Cleaning infinite and extreme values...")
    X = model_data.drop('TARGET', axis=1)
    y = model_data['TARGET']

    X = X.replace([np.inf, -np.inf], np.nan)

    for col in X.columns:
        if X[col].dtype in ['float64', 'float32', 'int64', 'int32']:
            valid = X[col].dropna()
            if len(valid) > 0:
                lower, upper = valid.quantile(0.001), valid.quantile(0.999)
                median = valid.median()
                X[col] = X[col].clip(lower=lower, upper=upper)
                X[col] = X[col].fillna(median)
            else:
                X[col] = X[col].fillna(0)

    print(f"✅ Data cleaned: {X.shape} - No infinite/NaN values")

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    print("\nData split:")
    print(f"  Train: {X_train.shape[0]:,} samples")
    print(f"  Test:  {X_test.shape[0]:,} samples")
    print(f"  Features: {X_train.shape[1]:,}")

    enhanced_model_results = {}

    # =============================
    # 1. XGBoost
    # =============================
    print("\n📊 Training XGBoost...")
    start = time.time()

    xgb_model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='auc',
        tree_method='hist',
        verbosity=0,
        missing=np.nan,
        reg_alpha=0.1,
        reg_lambda=0.1
    )

    xgb_model.fit(X_train, y_train)
    xgb_pred = xgb_model.predict_proba(X_test)[:, 1]
    xgb_auc = roc_auc_score(y_test, xgb_pred)
    xgb_time = time.time() - start

    enhanced_model_results['XGBoost'] = {
        'model': xgb_model,
        'auc_score': xgb_auc,
        'training_time': xgb_time,
        'predictions': xgb_pred
    }

    print(f"   ✅ XGBoost AUC: {xgb_auc:.4f} (trained in {xgb_time:.1f}s)")

    # =============================
    # 2. CatBoost
    # =============================
    if catboost_available:
        print("\n📊 Training CatBoost...")
        start = time.time()

        cb_model = CatBoostClassifier(
            iterations=100,
            depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bylevel=0.8,
            random_state=42,
            eval_metric='AUC',
            verbose=False,
            cat_features=original_categorical_cols if original_categorical_cols else None
        )

        cb_model.fit(X_train, y_train)
        cb_pred = cb_model.predict_proba(X_test)[:, 1]
        cb_auc = roc_auc_score(y_test, cb_pred)
        cb_time = time.time() - start

        enhanced_model_results['CatBoost'] = {
            'model': cb_model,
            'auc_score': cb_auc,
            'training_time': cb_time,
            'predictions': cb_pred
        }

        print(f"   ✅ CatBoost AUC: {cb_auc:.4f} (trained in {cb_time:.1f}s)")

    # =============================
    # 3. GradientBoosting
    # =============================
    print("\n📊 Training GradientBoostingClassifier...")
    start = time.time()

    gb_model = GradientBoostingClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        random_state=42,
        max_features='sqrt'
    )

    gb_model.fit(X_train, y_train)
    gb_pred = gb_model.predict_proba(X_test)[:, 1]
    gb_auc = roc_auc_score(y_test, gb_pred)
    gb_time = time.time() - start

    enhanced_model_results['GradientBoosting'] = {
        'model': gb_model,
        'auc_score': gb_auc,
        'training_time': gb_time,
        'predictions': gb_pred
    }

    print(f"   ✅ GradientBoosting AUC: {gb_auc:.4f} (trained in {gb_time:.1f}s)")

    # =============================
    # Results Summary
    # =============================
    print("\n" + "="*50)
    print("🎯 MODEL COMPARISON RESULTS")
    print("="*50)
    print(f"{'Model':<20} {'AUC Score':<10} {'Time (s)':<10} {'Rank':<6}")
    print("-" * 48)

    sorted_results = sorted(enhanced_model_results.items(), key=lambda x: x[1]['auc_score'], reverse=True)
    for rank, (model_name, res) in enumerate(sorted_results, 1):
        medal = "🥇" if rank == 1 else "🥈" if rank == 2 else "🥉"
        print(f"{model_name:<20} {res['auc_score']:<10.4f} {res['training_time']:<10.1f} {medal} #{rank}")

    best_model_name, best_res = sorted_results[0]
    print(f"\n🏆 CHAMPION: {best_model_name} with {best_res['auc_score']:.4f} AUC")

except Exception as e:
    print(f"❌ Error during model training: {e}")
    import traceback; traceback.print_exc()
    enhanced_model_results = None


# === FEATURE IMPORTANCE ANALYSIS ===
print("\n" + "="*80)
print("📈 FEATURE IMPORTANCE ANALYSIS")
print("="*80)

if enhanced_model_results:
    best_model_name = max(enhanced_model_results.items(), key=lambda x: x[1]['auc_score'])[0]
    best_model = enhanced_model_results[best_model_name]['model']

    print(f"Using {best_model_name} for feature importance analysis...")

    if hasattr(best_model, 'feature_importances_'):
        importances = best_model.feature_importances_
    else:
        print("❌ Model doesn't support feature importance")
        importances = None

    if importances is not None:
        feature_importance = pd.DataFrame({
            'feature': X.columns,
            'importance': importances
        }).sort_values('importance', ascending=False)

        print(f"\n🎯 TOP 20 MOST IMPORTANT FEATURES ({best_model_name}):")
        print("-" * 65)

        for i, (_, row) in enumerate(feature_importance.head(20).iterrows()):
            feature_type = (
                "📊 Bureau" if row['feature'].startswith('BUREAU_')
                else "🏪 POS" if ('POS_' in row['feature'] or 'PREV_' in row['feature'])
                else "📋 Application"
            )
            print(f"{i+1:2d}. {row['feature']:<40} {row['importance']:.4f} {feature_type}")

        top_20 = feature_importance.head(20)['feature'].tolist()
        bureau_count = sum(1 for f in top_20 if f.startswith('BUREAU_'))
        pos_count = sum(1 for f in top_20 if ('POS_' in f or 'PREV_' in f))
        app_count = 20 - bureau_count - pos_count

        print(f"\n📈 Feature Type Distribution in Top 20:")
        print(f"   📋 Application features: {app_count} ({app_count/20:.1%})")
        print(f"   🏪 POS features: {pos_count} ({pos_count/20:.1%})")
        print(f"   📊 Bureau features: {bureau_count} ({bureau_count/20:.1%})")

        bureau_features = [f for f in top_20 if f.startswith('BUREAU_')]
        if bureau_features:
            print(f"\n🔍 Top Bureau Risk Indicators:")
            for i, feat in enumerate(bureau_features[:5], 1):
                imp = feature_importance.loc[feature_importance['feature'] == feat, 'importance'].iloc[0]
                rank = top_20.index(feat) + 1
                print(f"   {i}. {feat:<35} {imp:.4f} (rank #{rank})")

        print(f"\n🎯 SUCCESS: Bureau risk features are proving their value!")
        print(f"   • {bureau_count} bureau features in top 20 ({bureau_count/20:.1%})")
        print(f"   • Best model: {best_model_name} ({enhanced_model_results[best_model_name]['auc_score']:.4f} AUC)")
        print(f"   • Enhanced dataset with {X.shape[1]} features ready for production!")

else:
    print("❌ Feature importance analysis not available")


In [ ]:
# === ENSEMBLE METHODS: MAJORITY VOTING & WEIGHTED AVERAGING ===
print("\n" + "=" * 80)
print("ENSEMBLE METHODS: MAJORITY VOTING & WEIGHTED AVERAGING")
print("=" * 80)

if enhanced_model_results and len(enhanced_model_results) >= 2:
    from sklearn.metrics import roc_auc_score, precision_recall_curve
    from sklearn.linear_model import LogisticRegression
    import numpy as np

    # Collect predictions and AUCs
    predictions = {m: r["predictions"] for m, r in enhanced_model_results.items()}
    model_aucs = {m: r["auc_score"] for m, r in enhanced_model_results.items()}

    print(f"Creating ensembles from {len(predictions)} models...")

    # 1) Simple Average
    print("\nMethod 1: Simple Average")
    pred_array = np.array(list(predictions.values()))
    simple_avg_pred = pred_array.mean(axis=0)
    simple_avg_auc = roc_auc_score(y_test, simple_avg_pred)
    print(f"   Simple Average AUC: {simple_avg_auc:.4f}")

    # 2) Weighted Average (by AUC)
    print("\nMethod 2: Weighted Average (AUC-based)")
    total_auc = sum(model_aucs.values())
    weights = {m: auc / total_auc for m, auc in model_aucs.items()}
    print("   Model weights:")
    for m, w in weights.items():
        print(f"      {m}: {w:.3f}")

    weighted_pred = np.zeros_like(simple_avg_pred)
    for m, pred in predictions.items():
        weighted_pred += weights[m] * pred
    weighted_auc = roc_auc_score(y_test, weighted_pred)
    print(f"   Weighted Average AUC: {weighted_auc:.4f}")

    # 3) Majority Voting (Hard)
    print("\nMethod 3: Majority Voting (Hard)")
    binary_preds, thresholds = {}, {}
    for m, pred in predictions.items():
        precision, recall, thresh = precision_recall_curve(y_test, pred)
        f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
        idx = np.argmax(f1_scores)
        thr = thresh[idx] if idx < len(thresh) else 0.5
        binary_preds[m] = (pred >= thr).astype(int)
        thresholds[m] = thr
        print(f"   {m} optimal threshold: {thr:.3f}")

    vote_array = np.array(list(binary_preds.values()))
    majority_vote = (vote_array.sum(axis=0) > vote_array.shape[0] / 2).astype(int)
    vote_confidence = vote_array.sum(axis=0) / vote_array.shape[0]
    majority_auc = roc_auc_score(y_test, vote_confidence)
    print(f"   Majority Voting AUC: {majority_auc:.4f}")

    # 4) Rank-Based Ensemble
    print("\nMethod 4: Rank-Based Ensemble")
    rank_preds = np.zeros_like(pred_array)
    for i, pred in enumerate(pred_array):
        rank_preds[i] = pred.argsort().argsort()
    avg_ranks = rank_preds.mean(axis=0)
    rank_auc = roc_auc_score(y_test, avg_ranks)
    print(f"   Rank-Based Ensemble AUC: {rank_auc:.4f}")

    # 5) Stacking (meta-learner on stacked probabilities)
    print("\nMethod 5: Stacking with Logistic Regression")
    stacking_features = np.column_stack(list(predictions.values()))
    meta_model = LogisticRegression(random_state=42, max_iter=1000)
    # Note: fitting on current split predictions; use OOF stacking in a full pipeline.
    meta_model.fit(stacking_features, y_test)
    stacking_pred = meta_model.predict_proba(stacking_features)[:, 1]
    stacking_auc = roc_auc_score(y_test, stacking_pred)
    print(f"   Stacking AUC: {stacking_auc:.4f}")
    print("   Meta-learner weights:")
    for i, m in enumerate(predictions.keys()):
        print(f"      {m}: {meta_model.coef_[0][i]:.3f}")

    # Results summary
    print("\n" + "=" * 60)
    print("ENSEMBLE RESULTS SUMMARY")
    print("=" * 60)

    ensemble_results = {
        "Simple Average": simple_avg_auc,
        "Weighted Average": weighted_auc,
        "Majority Voting": majority_auc,
        "Rank-Based": rank_auc,
        "Stacking": stacking_auc,
    }
    all_results = {**model_aucs, **ensemble_results}
    sorted_all = sorted(all_results.items(), key=lambda x: x[1], reverse=True)

    print(f"{'Method':<20} {'AUC Score':<12} {'Type':<15} {'Rank'}")
    print("-" * 55)
    for rank, (method, auc) in enumerate(sorted_all, 1):
        mtype = "Individual" if method in model_aucs else "Ensemble"
        print(f"{method:<20} {auc:<12.4f} {mtype:<15} #{rank}")

    best_method, best_auc = sorted_all[0]
    print(f"\nCHAMPION: {best_method} with {best_auc:.4f} AUC")

    best_individual = max(model_aucs.values())
    improvement = best_auc - best_individual
    rel_impr = (improvement / best_individual) if best_individual > 0 else float("nan")

    print("\nENSEMBLE IMPACT:")
    print(f"   Best individual model: {best_individual:.4f} AUC")
    print(f"   Best ensemble method: {best_auc:.4f} AUC")
    print(f"   Performance improvement: {improvement:+.4f} AUC ({rel_impr:.1%})")
    print("   Ensemble methods improving performance." if improvement > 0 else "   No improvement; models may be too similar.")

    # Save best ensemble predictions
    if best_method == "Simple Average":
        best_ensemble_pred = simple_avg_pred
    elif best_method == "Weighted Average":
        best_ensemble_pred = weighted_pred
    elif best_method == "Majority Voting":
        best_ensemble_pred = vote_confidence
    elif best_method == "Rank-Based":
        best_ensemble_pred = avg_ranks
    else:
        best_ensemble_pred = stacking_pred

    print("\nBest ensemble predictions saved for downstream use.")

else:
    print("Need at least 2 models in enhanced_model_results for ensemble methods.")


# Try with preprocessed credit card balance

In [ ]:
# === CREDIT CARD FEATURES INTEGRATION ===
print("=== INTEGRATING CREDIT CARD FEATURES ===")

# Load credit card features
print("Loading credit card features...")
try:
    credit_card_features = pl.read_csv('../data/credit_card_features.csv')
    print(f"✅ Credit card features loaded: {credit_card_features.shape}")
    print(f"   Customers with credit card history: {credit_card_features.shape[0]:,}")
    print(f"   Features: {credit_card_features.shape[1] - 1}")

    # Convert to pandas for integration
    credit_card_pandas = credit_card_features.to_pandas()

    # Display key features
    cc_features = [col for col in credit_card_pandas.columns if col != 'SK_ID_CURR']
    print(f"\n📊 Sample credit card features:")
    for feat in cc_features[:10]:
        print(f"   • {feat}")
    if len(cc_features) > 10:
        print(f"   ... and {len(cc_features) - 10} more features")

except Exception as e:
    print(f"❌ Error loading credit card features: {e}")
    print("Continuing without credit card features...")
    credit_card_pandas = pd.DataFrame({'SK_ID_CURR': []}) 

In [ ]:
# === MERGE WITH BASELINE + POS + BUREAU DATA ===
print("\n=== MERGING ALL FEATURE SETS ===")

# Start with existing baseline_pos_bureau data (includes SK_ID_CURR)
baseline_pos_bureau_cc = baseline_pos_bureau_pandas.copy()
print(f"Starting dataset shape: {baseline_pos_bureau_cc.shape}")

# Merge credit card features
if not credit_card_pandas.empty:
    print("Merging credit card features...")
    baseline_pos_bureau_cc = baseline_pos_bureau_cc.merge(
        credit_card_pandas,
        on="SK_ID_CURR",
        how="left"
    )
    print(f"After credit card merge: {baseline_pos_bureau_cc.shape}")

    # Count customers with/without credit card history
    cc_customers = baseline_pos_bureau_cc[credit_card_pandas.columns[1]].notna().sum()
    total_customers = baseline_pos_bureau_cc.shape[0]

    print(f"   Customers with credit card history: {cc_customers:,} "
          f"({cc_customers / total_customers * 100:.1f}%)")
    print(f"   Customers without credit card history: {total_customers - cc_customers:,} "
          f"({(total_customers - cc_customers) / total_customers * 100:.1f}%)")
else:
    print("⚠️  No credit card features to merge")

# Prepare final dataset for modeling
baseline_pos_bureau_cc_for_ml = baseline_pos_bureau_cc.drop("SK_ID_CURR", axis=1)

print("\n🎯 FINAL DATASET READY:")
print(f"   Shape: {baseline_pos_bureau_cc_for_ml.shape}")
print(f"   Features: {baseline_pos_bureau_cc_for_ml.shape[1] - 1} (excluding TARGET)")
print(f"   Missing values: {baseline_pos_bureau_cc_for_ml.isnull().sum().sum()}")

# Feature breakdown
original_features = baseline_pos_bureau_for_ml.shape[1] - 1  # excluding TARGET
if not credit_card_pandas.empty:
    cc_features_count = len([col for col in credit_card_pandas.columns if col != "SK_ID_CURR"])
    print(f"   Original features (baseline + POS + bureau): {original_features}")
    print(f"   Credit card features added: {cc_features_count}")
    print(f"   Total feature improvement: +{cc_features_count} features")

# Display sample of new features
print("\n📈 SAMPLE CREDIT CARD FEATURES IN DATASET:")
cc_cols = [col for col in baseline_pos_bureau_cc_for_ml.columns if col.startswith("CC_")]
for col in cc_cols[:8]:
    coverage = baseline_pos_bureau_cc_for_ml[col].notna().sum() / len(baseline_pos_bureau_cc_for_ml) * 100
    print(f"   • {col}: {coverage:.1f}% coverage")


In [ ]:
# === DATA CLEANING (FIXED VERSION) ===
print("\n🧹 Data cleaning...")

# Handle infinite values first
X = X.replace([np.inf, -np.inf], np.nan)

# Separate numeric and categorical columns
numeric_columns = X.select_dtypes(include=['float64', 'float32', 'int64',
'int32']).columns.tolist()
categorical_columns = X.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric columns: {len(numeric_columns)}")
print(f"Categorical columns: {len(categorical_columns)}")

# Handle categorical columns first
if categorical_columns:
    print(f"Encoding {len(categorical_columns)} categorical columns...")
    le = LabelEncoder()
    for col in categorical_columns:
        X[col] = X[col].fillna('Unknown')  # Fill missing values with 'Unknown'
        X[col] = le.fit_transform(X[col].astype(str))
        # Convert to numeric after encoding
        numeric_columns.append(col)

# Now clip extreme values for numeric columns only
for col in numeric_columns:
    if X[col].dtype in ['float64', 'float32', 'int64', 'int32']:
        valid_values = X[col].dropna()
        if len(valid_values) > 0:
            lower_bound = valid_values.quantile(0.001)
            upper_bound = valid_values.quantile(0.999)
            X[col] = X[col].clip(lower=lower_bound, upper=upper_bound)

# Fill missing values with median (now all columns are numeric)
X = X.fillna(X.median())

print(f"✅ Data cleaned. Final shape: {X.shape}")
print(f"✅ All columns are now numeric: {all(X.dtypes.apply(lambda x: x.kind in 'biufc'))}")

# === MODEL TRAINING WITH ENHANCED FEATURES ===
print(f"\n🚀 Training models with enhanced feature set ({X.shape[1]} features)...")

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

# Store results for comparison
enhanced_results = {}

# 1. Decision Tree
print("\n📊 Training Decision Tree...")
dt_enhanced = DecisionTreeClassifier(random_state=42, max_depth=10, min_samples_split=100)
dt_enhanced.fit(X_train, y_train)
dt_pred_enhanced = dt_enhanced.predict_proba(X_test)[:, 1]
dt_auc_enhanced = roc_auc_score(y_test, dt_pred_enhanced)
enhanced_results["Decision Tree"] = dt_auc_enhanced
print(f"Decision Tree AUC: {dt_auc_enhanced:.6f}")

# 2. XGBoost
print("\n📊 Training XGBoost...")
xgb_enhanced = XGBClassifier(random_state=42, eval_metric="logloss")
xgb_enhanced.fit(X_train, y_train)
xgb_pred_enhanced = xgb_enhanced.predict_proba(X_test)[:, 1]
xgb_auc_enhanced = roc_auc_score(y_test, xgb_pred_enhanced)
enhanced_results["XGBoost"] = xgb_auc_enhanced
print(f"XGBoost AUC: {xgb_auc_enhanced:.6f}")

# 3. CatBoost
print("\n📊 Training CatBoost...")
cat_enhanced = CatBoostClassifier(random_state=42, verbose=False)
cat_enhanced.fit(X_train, y_train)
cat_pred_enhanced = cat_enhanced.predict_proba(X_test)[:, 1]
cat_auc_enhanced = roc_auc_score(y_test, cat_pred_enhanced)
enhanced_results["CatBoost"] = cat_auc_enhanced
print(f"CatBoost AUC: {cat_auc_enhanced:.6f}")

# # 4. LightGBM (if available)
# try:
#     print("\n📊 Training LightGBM...")
#     lgb_enhanced = LGBMClassifier(random_state=42, verbose=-1)
#     lgb_enhanced.fit(X_train, y_train)
#     lgb_pred_enhanced = lgb_enhanced.predict_proba(X_test)[:, 1]
#     lgb_auc_enhanced = roc_auc_score(y_test, lgb_pred_enhanced)
#     enhanced_results["LightGBM"] = lgb_auc_enhanced
#     print(f"LightGBM AUC: {lgb_auc_enhanced:.6f}")
# except Exception as e:
#     print(f"⚠️  LightGBM not available: {e}")

print("\n✅ Enhanced model training completed!")


In [ ]:
baseline_pos_bureau_cc.to_csv('../data/baseline_pos_bureau_cc.csv', index=False)